### Question 8 (Hard) - Inventory manager with JSON and custom exceptions
Store an inventory in `inventory.json` as `{"item": {"qty": int, "price": float}, ...}`.

Write these functions:
* `load_inventory(path)` - if the file is **missing**, create it with `{}`; if it contains **invalid JSON**, rename it to `inventory.json.bak` and start with an empty inventory; handle `PermissionError` too.
* `save_inventory(path, data)` - write to a temporary file first and then replace the original with `os.replace` so a crash never leaves a half-written file.
* `add_item(inv, name, qty, price)` - rejects negative quantity/price with `ValueError`.
* `remove_stock(inv, name, qty)` - raises your own `ItemNotFoundError` or `OutOfStockError`.

Then print a report built with comprehensions: total inventory value (generator expression), low-stock items (`qty < 5`, dict comprehension), and the set of items priced above 500.

Test every failure path on purpose: missing file, corrupted file, removing a non-existent item, removing too much stock.

In [ ]:
import json
import os 

class ItmNotFounderror(Exception):
    """Raise when trying to modify an itm that does not exits in the inventory."""
    pass

class OutofstockError(Exception):
    """Raised when trying to reomve more stocks than is currently not available."""


# core inventory functions

def load_invetroy(path):
    "Load inventory from a json file.Handling missing, corrupted , or restriceted files"

    try:
        with open(path,'r')as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[load]file mising. creating a new empty inventory at'{path}.'") 
        inventory={}
        save_inve=(path,inventory)
        return save_inve
    except json.josde

In [ ]:

import json
import os 

class ItmNotFounderror(Exception):
    """Raise when trying to modify an item that does not exits in the inventory."""
    pass

class OutofstockError(Exception):
    """Raised when trying to remove more stocks than is currently not  available."""


# =====================================================================
# 2. Core Inventory Functions
# =====================================================================
def load_inventory(path):
    """Loads inventory from a JSON file. Handles missing, corrupted, or restricted files."""
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[Load] File missing. Creating a new empty inventory at '{path}'.")
        inventory = {}
        save_inventory(path, inventory)
        return inventory
    except json.JSONDecodeError:
        backup_path = f"{path}.bak"
        print(f"[Load] Invalid JSON detected. Renaming corrupted file to '{backup_path}'.")
        try:
            os.replace(path, backup_path)
        except PermissionError:
            print(f"[Error] Failed to create backup due to permission restrictions.")
        return {}
    except PermissionError:
        print(f"[Error] Permission denied while trying to read '{path}'. Returning empty inventory.")
        return {}

def save_inventory(path, data):
    """Atomically saves the inventory using a temporary file and os.replace."""
    tmp_path = f"{path}.tmp"
    try:
        # Write to temporary file first
        with open(tmp_path, 'w') as f:
            json.dump(data, f, indent=4)
        # Atomically swap the temporary file into the target path
        os.replace(tmp_path, path)
    except Exception as e:
        # Clean up temp file if something went wrong before the replace step
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise e

def add_item(inv, name, qty, price):
    """Adds or updates an item in the inventory dictionary."""
    if qty < 0 or price < 0:
        raise ValueError("Quantity and price cannot be negative.")
    
    if name in inv:
        inv[name]['qty'] += qty
        inv[name]['price'] = price  # Update to latest price
    else:
        inv[name] = {'qty': qty, 'price': price}

def remove_stock(inv, name, qty):
    """Deducts stock from an item or raises specific domain errors."""
    if name not in inv:
        raise ItemNotFoundError(f"Item '{name}' does not exist in inventory.")
    if inv[name]['qty'] < qty:
        raise OutOfStockError(f"Cannot remove {qty} units of '{name}'. Only {inv[name]['qty']} available.")
    
    inv[name]['qty'] -= qty

# =====================================================================
# 3. Reporting Logic
# =====================================================================
def print_report(inv):
    """Generates and prints an inventory report using comprehensions and generators."""
    print("\n--- INVENTORY REPORT ---")
    
    # Generator expression for total valuation
    total_value = sum(item['qty'] * item['price'] for item in inv.values())
    
    # Dict comprehension for low stock (< 5)
    low_stock = {name: item['qty'] for name, item in inv.items() if item['qty'] < 5}
    
    # Set comprehension for luxury items (> 500)
    expensive_items = {name for name, item in inv.items() if item['price'] > 500}
    
    print(f"Total Inventory Value: ${total_value:,.2f}")
    print(f"Low Stock Items (< 5 units): {low_stock}")
    print(f"Items Priced Over $500: {expensive_items}")
    print("------------------------\n")

# =====================================================================
# 4. Mandatory Failure Path & Behavior Testing
# =====================================================================
if __name__ == "__main__":
    db_path = "inventory.json"
    
    # Ensure environment is clean before testing
    for f in [db_path, f"{db_path}.bak", f"{db_path}.tmp"]:
        if os.path.exists(f):
            os.remove(f)

    # --- Test Path 1: Missing File ---
    print(">>> Test Path 1: Loading missing file...")
    inv = load_inventory(db_path) 
    print(f"Inventory current state: {inv}\n")

    # Seed initial working inventory
    add_item(inv, "Laptop", 10, 1200.00)
    add_item(inv, "Mouse", 3, 25.00)
    add_item(inv, "Keyboard", 2, 85.00)
    add_item(inv, "Monitor", 6, 600.00)
    save_inventory(db_path, inv)

    # --- Test Path 2: Corrupted JSON File ---
    print(">>> Test Path 2: Corrupting JSON file manually and reloading...")
    with open(db_path, 'w') as f:
        f.write("{ INVALID JSON CORRUPTION ...")
    
    inv = load_inventory(db_path)
    print(f"Inventory restarted state: {inv}\n")

    # Re-populate inventory for operational tests
    add_item(inv, "Laptop", 10, 1200.00)
    add_item(inv, "Mouse", 3, 25.00)
    add_item(inv, "Keyboard", 2, 85.00)
    add_item(inv, "Monitor", 6, 600.00)
    
    # --- Test Path 3: Negative Input Validation ---
    print(">>> Test Path 3: Adding negative values...")
    try:
        add_item(inv, "InvalidItem", -5, 10.00)
    except ValueError as e:
        print(f"Caught expected ValueError: {e}\n")

    # --- Test Path 4: Removing Non-existent Item ---
    print(">>> Test Path 4: Removing an item that doesn't exist...")
    try:
        remove_stock(inv, "Headphones", 1)
    except ItemNotFoundError as e:
        print(f"Caught expected ItemNotFoundError: {e}\n")

    # --- Test Path 5: Removing Too Much Stock ---
    print(">>> Test Path 5: Demanding more stock than available...")
    try:
        remove_stock(inv, "Keyboard", 5)
    except OutOfStockError as e:
        print(f"Caught expected OutOfStockError: {e}\n")

    # Final success operations & report execution
    print(">>> Simulating a valid stock reduction...")
    remove_stock(inv, "Laptop", 2)
    save_inventory(db_path, inv)
    
    print_report(inv)
